# Content-based baseline: TF-IDF

Baseline ini mengikuti protokol evaluasi yang sama dengan popularity dan matrix factorization (MF): model hanya melihat train, rekomendasi berasal dari warm candidate set, item yang sudah dilihat saat train dikeluarkan, dan relevansi validation memakai rating ? 4. Dataset tidak menyediakan deskripsi item, jadi representasi TF-IDF memakai user yang berinteraksi dengan item sebagai term. Karena itu, ini adalah baseline berbasis pola interaksi (kolaboratif tersirat), bukan kesamaan teks atau atribut item.

In [1]:
import sys
from pathlib import Path
import importlib.util
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from data_loader import load_data, relevant_items_val

# Load the project file directly to avoid a stale or conflicting content_based module in the kernel.
content_based_path = ROOT / 'src' / 'content_based.py'
spec = importlib.util.spec_from_file_location('project_content_based', content_based_path)
content_based = importlib.util.module_from_spec(spec)
spec.loader.exec_module(content_based)
TfidfContentRecommender = content_based.TfidfContentRecommender
build_item_docs = content_based.build_item_docs

train, val, cand_set, seen = load_data(str(ROOT / 'data') + '/')
relevant = relevant_items_val(val, cand_set, seen, min_rating=4)
print(f'train={len(train):,}; validation={len(val):,}; candidates={len(cand_set):,}; evaluable users={len(relevant):,}')

train=427,957; validation=41,284; candidates=22,753; evaluable users=11,850


## Item representation

Setiap item direpresentasikan sebagai dokumen berisi user ID dari interaksi train. TF-IDF memberi bobot lebih kecil pada user yang muncul di banyak dokumen; min_df=2 membuang user yang hanya muncul pada satu item. Seluruh vocabulary dan bobot dipelajari dari train saja.

In [2]:
item_docs = build_item_docs(train)
example_item = next(iter(item_docs))
tokens = item_docs[example_item].split()
print(f'items with documents: {len(item_docs):,}')
print(f'example item: {example_item}; {len(tokens)} user terms, e.g. {tokens[:3]}')

items with documents: 22,753
example item: 0000098906; 6 user terms, e.g. ['AGR5HW7AXRUCC4PZ6PTY454OXKMQ', 'AEPNK6CVNXCCF4WVPE63TD56KRLA', 'AG7FBPBRDJKCX2PAMCVJN26WQOPQ']


## Fit model

Profil user adalah rata-rata vektor TF-IDF item yang diberi rating ? 4 di train. Skor kandidat adalah dot product profil dengan vektor item; karena vektor item L2-normalized dan skala profil konstan untuk satu user, pengurutan setara dengan cosine similarity terhadap profil user. User tanpa profil dan kandidat tanpa sinyal positif diisi mengikuti urutan popularitas item dengan rating train ? 4.

In [3]:
model = TfidfContentRecommender(
    min_df=2, max_df=1.0, sublinear_tf=True,
    doc_min_rating=None, min_rating=4,
).fit(train, cand_set)
print(f'Fit complete: {len(model.item_index):,} items, vocab={len(model.vectorizer.vocabulary_):,} user terms, users with profile={len(model.user_items):,}')

Fit complete: 22,753 items, vocab=53,235 user terms, users with profile=54,036


## Evaluate ranking

Recall@10 dan NDCG@10 dihitung sebagai rata-rata per user yang memiliki setidaknya satu item relevan di validation. Definisi relevansi identik dengan baseline popularity dan MF: rating ? 4, item termasuk warm candidates, dan user memiliki interaksi train. Rekomendasi mengecualikan seluruh item yang sudah dilihat user di train.

In [4]:
def evaluate_at_k(model, relevant, seen, k=10):
    recalls, ndcgs = [], []
    for user_id, relevant_items in relevant.items():
        recs = model.recommend(user_id, seen, k=k)
        hits = [item in relevant_items for item in recs]
        recalls.append(sum(hits) / len(relevant_items))
        dcg = sum(hit / np.log2(rank + 2) for rank, hit in enumerate(hits))
        ideal_hits = min(len(relevant_items), k)
        idcg = sum(1 / np.log2(rank + 2) for rank in range(ideal_hits))
        ndcgs.append(dcg / idcg if idcg else 0.0)
    return {f'Recall@{k}': float(np.mean(recalls)), f'NDCG@{k}': float(np.mean(ndcgs))}

metrics = evaluate_at_k(model, relevant, seen, k=10)
metrics

{'Recall@10': 0.009305179263207244, 'NDCG@10': 0.005738041793491553}

## Inspect recommendations

Contoh rekomendasi dan item relevan validation untuk satu user evaluable. Metrik di atas menjadi hasil utama untuk dibandingkan dengan popularity dan MF pada split serta candidate set yang sama.

In [5]:
if relevant:
    example_user = next(iter(relevant))
    print('user:', example_user)
    print('recommendations:', model.recommend(example_user, seen, k=10))
    print('relevant validation items:', relevant[example_user])
else:
    print('No evaluable users in validation.')

user: AE22B7BMXT7GAGY3XLHICMESNAPA
recommendations: ['B01ATNCUHI', 'B07TV9WHMW', 'B09XXG98XF', 'B074MDDDLH', 'B07QHZYSHS', 'B084ZCV4Z8', 'B0C2PY3PJV', 'B07T73BTKC', 'B0BZ5BMQ5P', 'B07JLFRBN8']
relevant validation items: {'B0BYNM15YB'}
